# Fake News Detection - End-to-End NLP Project

A small classifier that reads a news headline and decides if it is fake (0) or real (1). The whole pipeline lives in this one notebook so it is easy to follow from start to finish.

### The pipeline at a glance

```
raw csv  ->  clean  ->  preprocess  ->  vectorise  ->  train  ->  evaluate  ->  predict
```


## Setup

All the imports we share across the notebook. If you need something specific to your section, add it inside that section.

In [ ]:
# ============================================================
# Setup - one import cell shared across the whole notebook
# ============================================================

# Standard library: regex + string constants used during text cleaning
import re
import string

# Core numerical / plotting stack
import numpy as np                 # vectorised math
import pandas as pd                # dataframes
import matplotlib.pyplot as plt    # plotting
import seaborn as sns              # prettier stats plots (heatmaps, hists)
import pickle                      # serialise the trained pipeline to disk

# NLTK - classic NLP toolkit. We use:
#   - stopwords: list of common English words to drop
#   - WordNetLemmatizer: maps word forms to a canonical lemma (running -> run)
#   - word_tokenize: splits a string into tokens (more robust than .split())
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# scikit-learn building blocks for the modelling pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
# Vectorisers turn text -> sparse numeric matrices
#   CountVectorizer  = Bag-of-Words (raw counts)
#   TfidfVectorizer  = TF-IDF (counts re-weighted by inverse document frequency)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
# Four classifiers we compare
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
# Pipeline chains (vectoriser + classifier) so there is no leakage between
# train/val and we can reuse the whole thing with a single .predict() call.
from sklearn.pipeline import Pipeline
# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

import joblib  # alternative serialisation (efficient for large numpy arrays)

# First-time setup: uncomment and run once to download NLTK data, then re-comment.
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')

# Fixed seed so every train/val split, tree, and SVM initialisation is reproducible.
RANDOM_STATE = 42
# Show full headlines in pandas previews instead of truncating them.
pd.set_option('display.max_colwidth', 120)


---
# 1. The data

Two files in `dataset/`:

- `training_data_lowercase.csv` - has `label` and `headline`. Labels are 0 (fake) or 1 (real). This is what we train on.
- `testing_data_lowercase_nolabels.csv` - just `headline`. No labels - this is what we have to predict.

---
# 2. Loading and cleaning the data

In [ ]:
# ------------------------------------------------------------
# Load the two CSV files. Paths are relative to the notebook.
# ------------------------------------------------------------
TRAIN_PATH = 'dataset/training_data_lowercase.csv'
TEST_PATH = 'dataset/testing_data_lowercase_nolabels.csv'

# The source files are tab-separated and have NO header row, so we must:
#   - set sep='\t'        (tab delimiter)
#   - set header=None     (first row is real data, not column names)
#   - supply names=[...]  (we name the columns ourselves)
# Train has (label, headline); test has only (headline) - labels are what we predict.
df_train = pd.read_csv(TRAIN_PATH, sep='\t', header=None, names=['label', 'headline'])
df_test = pd.read_csv(TEST_PATH, sep='\t', header=None, names=['headline'])

# Sanity check: row counts and a quick preview of the training data.
print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)
df_train.head()


In [ ]:
# ------------------------------------------------------------
# Data-quality checks BEFORE modelling.
# ------------------------------------------------------------

# 1) Missing values: text classifiers don't handle NaN, so catch them early.
print('Missing in train:')
print(df_train.isna().sum())
print('\nMissing in test:')
print(df_test.isna().sum())

# 2) Duplicate headlines inflate both training size and validation accuracy
#    (the same row could end up in both sets). Dropping them gives an
#    honest evaluation - this is the single biggest quality win from cleaning.
n_dup = df_train.duplicated().sum()
print(f'\nDuplicate rows in train: {n_dup}')
df_train = df_train.drop_duplicates().reset_index(drop=True)

# 3) Enforce dtypes: labels must be int for sklearn, headlines must be str so
#    NLTK / vectorisers don't choke on mixed types.
df_train['label'] = df_train['label'].astype(int)
df_train['headline'] = df_train['headline'].astype(str)
df_test['headline'] = df_test['headline'].astype(str)

# Final summary: row count, dtypes, memory footprint.
df_train.info()


## 2.1 A quick look at the data (EDA)

In [ ]:
# ------------------------------------------------------------
# EDA plot 1: class balance.
# If classes are very imbalanced, plain accuracy becomes misleading
# (a dummy classifier that always predicts the majority class looks "good").
# Here it is roughly 52% real / 48% fake, so accuracy is a fair headline metric.
# ------------------------------------------------------------
ax = df_train['label'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'])
ax.set_xticklabels(['Fake (0)', 'Real (1)'], rotation=0)
ax.set_title('Class distribution')
plt.show()

# ------------------------------------------------------------
# EDA plot 2: headline length distribution, split by class.
# Why we care:
#   (a) typical length informs max_features / ngram_range choices later;
#   (b) if one class has systematically longer/shorter headlines, length
#       itself becomes a weak discriminative signal worth knowing about.
# ------------------------------------------------------------
df_train['n_words'] = df_train['headline'].str.split().str.len()
sns.histplot(data=df_train, x='n_words', hue='label', bins=40, kde=True)
plt.title('Headline length (words) by class')
plt.show()


---
# 3. Preprocessing

In [ ]:
# ------------------------------------------------------------
# clean_text(): the single preprocessing function shared across the project.
# It must be reused everywhere - NO custom preprocessing elsewhere,
# otherwise train / test features would diverge.
# ------------------------------------------------------------

# Pre-compute these once at module load time (cheaper than rebuilding in-loop):
STOPWORDS = set(stopwords.words('english'))          # O(1) membership lookup
LEMMATIZER = WordNetLemmatizer()                     # stateful lemmatiser
# Single regex matching every punctuation character AND every digit.
# Compiled once so each clean_text() call doesn't re-parse the pattern.
PUNCT_RE = re.compile(f'[{re.escape(string.punctuation)}0-9]')


def clean_text(text: str) -> str:
    """Lowercase, strip punctuation/digits, drop stopwords, lemmatise."""
    # 1) Lowercase so "Trump" and "trump" collapse to one token.
    text = text.lower()
    # 2) Replace punctuation and digits with spaces - keeps word boundaries intact.
    text = PUNCT_RE.sub(' ', text)
    # 3) Tokenise with NLTK (handles contractions and edge cases .split() misses).
    tokens = word_tokenize(text)
    # 4) Drop stopwords ("the", "and", ...) and single-character noise.
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    # 5) Lemmatise: "running" -> "run". We chose lemmatisation over stemming
    #    so the SVM's coefficient words stay human-readable for interpretation.
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    # 6) Re-join into a single whitespace-separated string - that's what
    #    CountVectorizer / TfidfVectorizer expect as input.
    return ' '.join(tokens)


# Quick smoke test on a deliberately noisy example.
clean_text("Donald Trump sends out EMBARRASSING tweets!!! #fakenews")


In [ ]:
# Apply clean_text() to every headline in BOTH dataframes, storing the
# result in a new 'processed_text' column so we can still compare raw vs clean.
# .apply() runs clean_text once per row - plenty fast enough at ~32k rows.
df_train['processed_text'] = df_train['headline'].apply(clean_text)
df_test['processed_text'] = df_test['headline'].apply(clean_text)

# Show raw headline, cleaned version, and label side-by-side for a sanity check.
df_train[['headline', 'processed_text', 'label']]


### 3.1 Top words per class

A quick look at the most frequent words in each class. Useful sanity check that cleaning didn't strip the signal.

In [ ]:
# ------------------------------------------------------------
# Top words per class (EDA continued).
# This is a sanity check: cleaning should NOT have stripped the
# discriminative signal. If fake / real top-20 lists look indistinguishable,
# the model will struggle regardless of which algorithm we pick.
# ------------------------------------------------------------
from collections import Counter

# Concatenate every cleaned headline per class into one big string, then
# split back into tokens - Counter will give us the 20 most frequent.
fake_words = ' '.join(df_train.loc[df_train['label'] == 0, 'processed_text']).split()
real_words = ' '.join(df_train.loc[df_train['label'] == 1, 'processed_text']).split()

top_n = 20
fake_counts = Counter(fake_words).most_common(top_n)
real_counts = Counter(real_words).most_common(top_n)

# Side-by-side horizontal bars, red for fake, green for real.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# [::-1] reverses the lists so the largest count is at the top of the chart.
words, counts = zip(*fake_counts)
axes[0].barh(list(words)[::-1], list(counts)[::-1], color='#e74c3c')
axes[0].set_title(f'Top {top_n} words in FAKE headlines', fontsize=12)
axes[0].set_xlabel('Frequency')

words, counts = zip(*real_counts)
axes[1].barh(list(words)[::-1], list(counts)[::-1], color='#2ecc71')
axes[1].set_title(f'Top {top_n} words in REAL headlines', fontsize=12)
axes[1].set_xlabel('Frequency')

plt.tight_layout()
# Save so the chart can be embedded in the final presentation deck.
plt.savefig('top_words_by_class.png', dpi=150)
plt.show()


---
# 4. Training the models

## Train / Test Split

In [ ]:
# ------------------------------------------------------------
# Train / validation split.
# 80% train / 20% validation - the 20% is held out until the very end
# so accuracy numbers aren't inflated by data the model has already seen.
# random_state=42 makes the split reproducible across reruns.
# ------------------------------------------------------------

# Target labels (0 = Fake, 1 = Real)
y_target = df_train['label']

# We keep TWO parallel splits - one of raw headlines, one of cleaned text -
# so we can measure the actual impact of preprocessing on the same folds.
# Both splits use the SAME random_state so y_train/y_test align perfectly.

# Split 1: raw headlines (no cleaning applied)
Xraw_train, Xraw_test, y_train, y_test = train_test_split(
    df_train['headline'], y_target, test_size=0.20, random_state=42
)

# Split 2: cleaned headlines (after clean_text). y's are captured with _ since
# they are identical to the ones above - same random_state.
Xclean_train, Xclean_test, _, _ = train_test_split(
    df_train['processed_text'], y_target, test_size=0.20, random_state=42
)

# Convenience aliases: downstream evaluation cells expect X_train / X_test.
X_train, X_test = Xclean_train, Xclean_test

print(f"-> Training set: {len(Xraw_train)} samples")
print(f"-> Test set: {len(Xraw_test)} samples")


---
# 5. Feature engineering

## Defining pipelines

In [ ]:
# ------------------------------------------------------------
# Define 8 experiment pipelines = 4 classifiers x 2 text representations
# (each classifier is tried on both RAW and CLEAN text).
#
# Each tuple is (name, Pipeline, X_train, X_val) so the training loop can
# stay generic - one loop handles all 8 combinations.
#
# Using sklearn.Pipeline means the vectoriser is .fit()'d ONLY on train data;
# when we call pipeline.predict(X_val) it transforms validation text with the
# same vocabulary. That's how we avoid lookahead leakage on TF-IDF IDF weights.
# ------------------------------------------------------------
experiments = [
    # 1. Multinomial Naive Bayes + Bag-of-Words (a classic text-classification baseline)
    ("Raw + BoW + Naive Bayes",   Pipeline([('vec', CountVectorizer()), ('clf', MultinomialNB())]), Xraw_train, Xraw_test),
    ("Clean + BoW + Naive Bayes", Pipeline([('vec', CountVectorizer()), ('clf', MultinomialNB())]), Xclean_train, Xclean_test),

    # 2. Logistic Regression + TF-IDF (strong linear baseline; max_iter bumped
    #    so it actually converges on sparse high-dim text)
    ("Raw + TF-IDF + LR", Pipeline([('vec', TfidfVectorizer()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]), Xraw_train, Xraw_test),
    ("Clean + TF-IDF + LR", Pipeline([('vec', TfidfVectorizer()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]), Xclean_train, Xclean_test),

    # 3. Random Forest + TF-IDF (non-linear, struggles on sparse high-dim text
    #    but included for contrast; n_jobs=-1 uses all cores)
    ("Raw + TF-IDF + Random Forest", Pipeline([('vec', TfidfVectorizer()), ('clf', RandomForestClassifier(random_state=42, n_jobs=-1))]), Xraw_train, Xraw_test),
    ("Clean + TF-IDF + Random Forest", Pipeline([('vec', TfidfVectorizer()), ('clf', RandomForestClassifier(random_state=42, n_jobs=-1))]), Xclean_train, Xclean_test),

    # 4. Linear SVM + TF-IDF (usually the winner on short-text classification)
    ("Raw + TF-IDF + SVM", Pipeline([('vec', TfidfVectorizer()), ('clf', LinearSVC(random_state=42))]), Xraw_train, Xraw_test),
    ("Clean + TF-IDF + SVM", Pipeline([('vec', TfidfVectorizer()), ('clf', LinearSVC(random_state=42))]), Xclean_train, Xclean_test)
]


## Train and evaluate Pipelines, check overfitting

In [ ]:
# ------------------------------------------------------------
# Train every pipeline and record Train vs Validation accuracy.
# The gap between the two is our overfitting signal:
#   - Small gap (e.g. +2 pts)  -> model generalises well
#   - Huge gap  (e.g. +10 pts) -> model has memorised the training set
# ------------------------------------------------------------
print("Testing combinations pipelines (Checking Train vs Test Accuracy)...\n")

# Collect results into parallel lists so we can plot them afterwards.
results_names = []
train_accuracies = []
val_accuracies = []
best_baseline_acc = 0  # running max, so we know which pipeline wins

for name, pipeline, X_train_set, X_val_set in experiments:
    print(f"Training {name}...")
    # .fit() vectorises the training text AND fits the classifier in one step.
    pipeline.fit(X_train_set, y_train)

    # Training accuracy: how well the model memorised what it was shown.
    train_acc = accuracy_score(y_train, pipeline.predict(X_train_set))
    # Validation accuracy: how well it generalises to held-out headlines.
    val_acc = accuracy_score(y_test, pipeline.predict(X_val_set))

    # Store for the comparison chart
    results_names.append(name)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"   Score -> Train: {train_acc:.4f} | Val: {val_acc:.4f}")

    # Track the best validation score - that's the pipeline worth tuning.
    if val_acc > best_baseline_acc:
        best_baseline_acc = val_acc
        best_baseline_name = name

print(f"\n-----> BEST BASELINE COMBINATION: {best_baseline_name} ({best_baseline_acc:.4f})")


## Models Comparison chart
Comparing with/without preprocessing

In [ ]:
# ------------------------------------------------------------
# Comparison chart: Train vs Validation accuracy for all 8 pipelines.
# What to look for:
#   - Two close lines -> model generalises (SVM, NB)
#   - Train near 100% while Val much lower -> severe overfitting (Random Forest)
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))  # wide enough for 8 x-axis labels

# Plot training (blue circles) and validation (red squares) side by side.
ax.plot(results_names, train_accuracies, marker='o', color='#3498db', label='Training Accuracy', linewidth=2, markersize=8)
ax.plot(results_names, val_accuracies, marker='s', color='#e74c3c', label='Validation Accuracy', linewidth=2, markersize=8)

# y-axis starts at 0.85 so small differences between models are visible.
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title('Exhaustive Impact of Preprocessing & Model Selection', fontsize=14, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)

# Rotate labels so long pipeline names don't overlap.
plt.xticks(rotation=35, ha='right', fontsize=10)
ax.legend(loc='lower right', fontsize=11)

# Print the winner to console so the number is easy to copy into the slide deck.
best_idx = np.argmax(val_accuracies)
print(f"\nBEST CONFIGURATION: {results_names[best_idx]} with {val_accuracies[best_idx]*100:.2f}% Accuracy")

plt.tight_layout()
plt.savefig('group1_model_comparison.png', dpi=150)  # saved for the pptx deck
plt.show()


## Hyperparameter Tuning (GridSearchCV)
Optimizing our Best Model (Raw + TF-IDF + SVM)

In [ ]:
# ------------------------------------------------------------
# Hyperparameter tuning on the baseline winner (Raw + TF-IDF + SVM).
# We use GridSearchCV:
#   - tries every combination of the param_grid below,
#   - evaluates each with 5-fold cross-validation on the TRAIN set only,
#   - picks the combination with the best mean CV accuracy.
# Held-out X_test is never used during search, so tuning can't leak into
# the final validation number.
# ------------------------------------------------------------
pipe_to_tune = Pipeline([
    ("vec", TfidfVectorizer()),    # will be re-fit on each CV fold
    ("clf", LinearSVC(random_state=42))
])

# The 'stepname__param' syntax tells the Pipeline which component each
# hyperparameter belongs to (vec vs clf).
param_grid = {
    'vec__ngram_range': [(1, 1), (1, 2)],            # unigrams only vs uni+bigrams
    'vec__max_features': [None, 5000],               # no cap vs a small vocabulary
    'clf__C': [0.01, 0.05, 0.1, 0.5, 1, 10]          # SVM regularisation strength
}

print("Running GridSearchCV...")
# cv=5 -> 5-fold cross-validation; n_jobs=-1 parallelises across all CPU cores.
grid_search = GridSearchCV(pipe_to_tune, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(Xraw_train, y_train)

# grid_search.best_estimator_ is already refit on the FULL Xraw_train with the
# best hyperparameters. Evaluate it on the held-out validation set.
tuned_test_acc = accuracy_score(y_test, grid_search.predict(Xraw_test))

print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Tuned Model Validation Accuracy: {tuned_test_acc:.4f}")

# Rebuild an un-tuned baseline pipeline so the next cell can chart the lift.
baseline_pipe = Pipeline([("vec", TfidfVectorizer()), ("clf", LinearSVC(random_state=42))])
baseline_pipe.fit(Xraw_train, y_train)
baseline_acc = accuracy_score(y_test, baseline_pipe.predict(Xraw_test))


### VISUALIZATION: TUNING IMPACT

In [ ]:
# ------------------------------------------------------------
# Visualise the lift from hyperparameter tuning: baseline vs tuned SVM.
# Tight y-axis zoom (0.92 - 0.96) makes sub-percent differences readable.
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 5))

# Two bars: default-params pipeline vs GridSearchCV winner.
bars = ax.bar(['Baseline SVM\n(Default Params)', 'Tuned SVM\n(GridSearchCV)'],
              [baseline_acc, tuned_test_acc],
              color=['#34495e', '#f1c40f'],
              width=0.5)

ax.set_ylim(0.92, 0.96)  # zoom in so tenth-of-a-percent gains are visible
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Impact of Hyperparameter Tuning', fontsize=14, fontweight='bold')

# Annotate each bar with its exact percentage (matches the slide deck).
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.001, f"{yval*100:.2f}%",
            ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('group1_tuning_impact.png', dpi=150)  # embedded in slide 9 of the deck
plt.show()


## Export Pipeline

In [ ]:
# ------------------------------------------------------------
# Persist the tuned pipeline so the evaluation cells can load it
# instead of retraining. Pickle serialises the WHOLE pipeline (vectoriser
# vocabulary + SVM weights) into a single file.
# Rule: whoever loads this must .predict() only - never .fit() - on test data.
# ------------------------------------------------------------
final_best_model = grid_search.best_estimator_
output_filename = 'pipeline_best_model.pkl'

with open(output_filename, 'wb') as file:
    pickle.dump(final_best_model, file)

print(f"\nTuned pipeline successfully exported as saved as '{output_filename}'")


### 5.1 Most informative tokens (model interpretability)

Look at the SVM coefficients to see which words push the model toward FAKE vs REAL. A sanity check that the model is learning sensible features.

In [ ]:
# ------------------------------------------------------------
# Interpretability: which words/n-grams push the SVM toward FAKE vs REAL?
# A Linear SVM assigns one coefficient per feature; sign tells us the class
# direction, magnitude tells us the strength. If the top words look random,
# the model is picking up noise - here we expect style cues (reuters, says,
# video, watch) which is reassuring.
# ------------------------------------------------------------

# Pull the two steps out of the pipeline so we can access their internals.
vec = final_best_model.named_steps['vec']   # TfidfVectorizer (has vocabulary)
clf = final_best_model.named_steps['clf']   # LinearSVC (has coefficients)

# Parallel arrays: feature_names[i] <-> coefs[i]
feature_names = np.array(vec.get_feature_names_out())
coefs = clf.coef_.ravel()  # .ravel() flattens the 1xN matrix into a 1D array

# np.argsort returns indices that would sort the array ascending, so:
#   last 20  -> most POSITIVE coefs -> strongest REAL signal
#   first 20 -> most NEGATIVE coefs -> strongest FAKE signal
top_n = 20
top_real_idx = np.argsort(coefs)[-top_n:]
top_fake_idx = np.argsort(coefs)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(feature_names[top_fake_idx], coefs[top_fake_idx], color='#e74c3c')
axes[0].set_title(f'Top {top_n} features pushing toward FAKE')
axes[0].set_xlabel('Coefficient')

axes[1].barh(feature_names[top_real_idx], coefs[top_real_idx], color='#2ecc71')
axes[1].set_title(f'Top {top_n} features pushing toward REAL')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.savefig('top_coefficients.png', dpi=150)
plt.show()


---
# 6. Evaluation
Now that the model is trained, it is time to measure how well it actually works and understand where it makes mistakes.

Metrics computed here:
- **Accuracy** — overall % of correct predictions
- **Precision** — of the headlines predicted as Real, how many actually are?
- **Recall** — of all actual Real headlines, how many did we catch?
- **F1** — harmonic mean of Precision and Recall (best single-number summary)
- **Confusion Matrix** — visual breakdown of TP, TN, FP, FN

In [ ]:
# ------------------------------------------------------------
# Evaluation step 1: load the already-trained pipeline from disk.
# We INTENTIONALLY do not retrain here - evaluation must use the exact
# model that will be submitted, otherwise numbers drift between cells.
# ------------------------------------------------------------
with open('pipeline_best_model.pkl', 'rb') as f:
    model_pipeline = pickle.load(f)

print(" Model loaded successfully!")
print(f"Pipeline steps: {[step[0] for step in model_pipeline.steps]}")


In [ ]:
# ------------------------------------------------------------
# Evaluation step 2: predict on the held-out validation set.
# .predict() returns 0 (Fake) or 1 (Real) per headline. No .fit() is ever
# called here - the model's weights are frozen.
# ------------------------------------------------------------
y_pred = model_pipeline.predict(X_test)

print(f"Predictions generated: {len(y_pred)} samples")


In [ ]:
# ------------------------------------------------------------
# Evaluation step 3: the core metrics (all comparing y_pred vs y_test).
# Why four metrics instead of just accuracy?
#   - Accuracy can hide asymmetric errors (e.g. all FNs, no FPs).
#   - In fake-news detection the "Real=1" class is the positive class;
#     precision/recall tell us what kind of mistakes we make.
# ------------------------------------------------------------

# accuracy_score: (correct predictions) / (total predictions)
acc = accuracy_score(y_test, y_pred)

# precision_score: of the headlines we PREDICTED as Real (1), how many truly were?
# High precision = few false alarms (few fakes mis-labelled as real).
prec = precision_score(y_test, y_pred)

# recall_score: of the actual Real (1) headlines, how many did we catch?
# High recall = few misses (few real headlines mis-labelled as fake).
rec = recall_score(y_test, y_pred)

# f1_score: harmonic mean of precision + recall. Single scalar that penalises
# being great at one metric while terrible at the other.
f1 = f1_score(y_test, y_pred)

print("=" * 40)
print("EVALUATION RESULTS")
print("=" * 40)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1 Score  : {f1:.4f}")


In [ ]:
# ------------------------------------------------------------
# Evaluation step 4: full classification report.
# Shows per-class precision, recall, F1 and support (row count).
# Useful for spotting asymmetry - e.g. Fake recall 0.89 vs Real recall 0.97
# means the model is more cautious about labelling things as Real.
# ------------------------------------------------------------
print("CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Fake News (0)', 'Real News (1)']
))


In [ ]:
# ------------------------------------------------------------
# Evaluation step 5: confusion matrix visualisation.
# Rows = true label, columns = predicted label. Diagonal = correct.
# Off-diagonal cells are the errors split by type:
#   top-right    (True Fake, Pred Real) = False Positive
#   bottom-left  (True Real, Pred Fake) = False Negative
# ------------------------------------------------------------

# confusion_matrix returns a 2x2 array of counts (TN, FP / FN, TP).
cm = confusion_matrix(y_test, y_pred)

# Use seaborn for a coloured heatmap with count labels.
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,                          # write the count into each cell
    fmt='d',                             # format as plain integer
    cmap='Blues',                        # blue gradient (higher = darker)
    xticklabels=['Fake (0)', 'Real (1)'],
    yticklabels=['Fake (0)', 'Real (1)'],
    ax=ax
)
ax.set_title('Confusion Matrix - Best Model', fontsize=13)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)  # goes into slide 11 of the deck
plt.show()
print(" Saved as 'confusion_matrix.png'")


### 6.1 Threshold tuning (decision-function score)

LinearSVC has no probability output; use the decision function as a continuous score and sweep the threshold to see where precision/recall/F1 trade off.

In [ ]:
# ------------------------------------------------------------
# Threshold tuning on the decision function.
# LinearSVC has no predict_proba, but decision_function() returns a signed
# score - negative -> Fake, positive -> Real - and .predict() is equivalent
# to (score >= 0). By sweeping the threshold we can see how precision, recall
# and F1 trade off, and whether the default threshold (0) is F1-optimal.
# ------------------------------------------------------------
scores = model_pipeline.decision_function(X_test)

# 200 evenly spaced thresholds between the min and max observed score.
thresholds = np.linspace(scores.min(), scores.max(), 200)
precisions, recalls, f1s = [], [], []

# For each candidate threshold, re-binarise predictions and recompute metrics.
for t in thresholds:
    preds = (scores >= t).astype(int)
    precisions.append(precision_score(y_test, preds, zero_division=0))
    recalls.append(recall_score(y_test, preds, zero_division=0))
    f1s.append(f1_score(y_test, preds, zero_division=0))

# Argmax of F1 = the single threshold that maximises balanced performance.
best_idx = int(np.argmax(f1s))
best_t = thresholds[best_idx]

plt.figure(figsize=(10, 5))
plt.plot(thresholds, precisions, label='Precision', color='#3498db')
plt.plot(thresholds, recalls, label='Recall', color='#e67e22')
plt.plot(thresholds, f1s, label='F1', color='#2ecc71', linewidth=2)
# Dashed reference line at the default threshold (0).
plt.axvline(0, color='gray', linestyle='--', alpha=0.5, label='Default threshold (0)')
# Dashed marker at the F1-optimal threshold found above.
plt.axvline(best_t, color='#9b59b6', linestyle='--', alpha=0.7,
            label=f'Best F1 @ {best_t:.2f}')
plt.title('Precision / Recall / F1 vs decision threshold')
plt.xlabel('Decision function threshold')
plt.ylabel('Score')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig('threshold_tuning.png', dpi=150)
plt.show()

print(f'Best F1 at threshold {best_t:.3f} -> F1 = {f1s[best_idx]:.4f}')
print(f'(Default threshold 0 gave F1 = {f1_score(y_test, (scores >= 0).astype(int)):.4f})')


In [ ]:
# ------------------------------------------------------------
# Evaluation step 6: error analysis.
# Metrics tell us HOW OFTEN we're wrong; error analysis tells us WHERE.
# Filtering for misclassified rows lets us eyeball patterns that might
# warrant better features, more data, or a threshold shift.
# ------------------------------------------------------------
eval_df = pd.DataFrame({
    'headline' : X_test.values,    # cleaned text fed to the model
    'true'     : y_test.values,    # correct label (0 or 1)
    'predicted': y_pred            # what the model said
})

# Keep only the rows where the model was wrong.
misclassified = eval_df[eval_df['true'] != eval_df['predicted']]

print(f"Total misclassifications: {len(misclassified)} out of {len(eval_df)} "
      f"({len(misclassified)/len(eval_df)*100:.1f}%)")
print()


In [ ]:
# ------------------------------------------------------------
# False Negatives = real headlines the model dismissed as fake.
# Reading 5 of them is usually enough to spot a theme (e.g. opinion pieces,
# sarcasm, non-standard attribution) that we could fix with better features.
# ------------------------------------------------------------
false_negatives = misclassified[misclassified['true'] == 1].head(5)
print("=== Real headlines the model thought were FAKE (False Negatives) ===")
for _, row in false_negatives.iterrows():
    print(f"  '{row['headline']}'")
print()


In [ ]:
# ------------------------------------------------------------
# False Positives = fake headlines the model accepted as real.
# These are the scarier errors in a fake-news context: fake content gets
# the "real" label. If this list shows a pattern we'd bias the threshold
# toward higher precision (fewer false positives, at the cost of recall).
# ------------------------------------------------------------
false_positives = misclassified[misclassified['true'] == 0].head(5)
print("=== Fake headlines the model thought were REAL (False Positives) ===")
for _, row in false_positives.iterrows():
    print(f"  '{row['headline']}'")


---
# 7. Predicting on the test set
Now we apply the model to the real test set — the one with no labels.
This is what we submit.

Key rules:
- Use `clean_text()` on the test headlines (same preprocessing as training).
- Use the already-fitted `model_pipeline` — **no refitting**.
- Only call `.predict()` — never `.fit()` on test data.

In [ ]:
# ------------------------------------------------------------
# Prediction step 1: sanity-check the unlabelled test set is loaded and
# cleaned the SAME way as training data.
# processed_text column was populated way back in cell 11 using clean_text().
# ------------------------------------------------------------
print(f"Test set size: {len(df_test)} headlines")
print()
print("Preview of test data:")
df_test[['headline', 'processed_text']].head()


In [ ]:
# ------------------------------------------------------------
# Prediction step 2: run the frozen pipeline on every test headline.
# One .predict() call per row, returning 0 (Fake) or 1 (Real).
# Then print the class distribution as a sanity check - if it's 99/1,
# something has gone wrong (e.g. feature mismatch between train and test).
# ------------------------------------------------------------
test_predictions = model_pipeline.predict(df_test['processed_text'])

unique, counts = np.unique(test_predictions, return_counts=True)
print(f"Predictions generated: {len(test_predictions)}")
print()
for label, count in zip(unique, counts):
    name = 'Fake' if label == 0 else 'Real'
    print(f"  Predicted {name} ({label}): {count} headlines ({count/len(test_predictions)*100:.1f}%)")


---
# 8. Final output
Two deliverables:
1. `submission.csv` — predictions for every test headline, in order
2. `model_pipeline.joblib` — the saved pipeline for reproducibility

In [ ]:
# ------------------------------------------------------------
# Deliverable 1: submission CSV.
# Two columns - prediction (0/1) and the original headline - one row per
# test headline, kept in the original order so downstream graders can
# align rows without needing an ID column.
# ------------------------------------------------------------
submission = pd.DataFrame({'prediction': test_predictions})
submission['headline'] = df_test['headline'].values

submission.to_csv('G1.csv', index=False)  # index=False -> no extra row-number column

print(" G1.csv saved!")
print()
print("Preview:")
print(submission.head(10))


In [ ]:
# ------------------------------------------------------------
# Deliverable 2: the trained pipeline, serialised with joblib.
# joblib is preferred over pickle for sklearn objects because it handles
# large numpy arrays (model weights, vectoriser vocab) more efficiently.
# Anyone can now `joblib.load('model_pipeline.joblib')` and .predict() on
# new headlines without retraining.
# ------------------------------------------------------------
joblib.dump(model_pipeline, 'model_pipeline.joblib')

print(" model_pipeline.joblib saved!")


In [ ]:
# ------------------------------------------------------------
# Final summary block - one-stop snapshot of dataset sizes, model choice,
# validation metrics and the files we're handing off. These are the exact
# numbers that appear on the slide deck, so any change here ripples to the
# presentation via doc/update_merged_pptx.py.
# ------------------------------------------------------------
print("=" * 55)
print("FAKE NEWS DETECTION - FINAL SUMMARY")
print("=" * 55)
print()
print("Dataset")
print(f"  Training samples   : {len(X_train)}")
print(f"  Validation samples : {len(X_test)}")
print(f"  Test samples       : {len(df_test)}")
print()
print("Best Model")
print(f"  Pipeline           : TF-IDF + Linear SVM + RAW (tuned with GridSearchCV)")
print(f"  Validation Accuracy: {tuned_test_acc*100:.2f}%")
print(f"  Precision          : {prec:.4f}")
print(f"  Recall             : {rec:.4f}")
print(f"  F1 Score           : {f1:.4f}")
print()
print("Output files")
print("   submission.csv         -> Test set predictions (0=Fake, 1=Real)")
print("   model_pipeline.joblib  -> Saved model for reproducibility")
print("   confusion_matrix.png   -> Confusion matrix chart")


---
# 9. Wrap-up and ideas for later

What we ended up with: a small, reproducible pipeline that cleans headlines, turns them into TF-IDF features, runs them through a classifier, and spits out predictions for the unlabelled test set.

Things we could try if we had more time:
- Word embeddings (Word2Vec, GloVe) instead of TF-IDF.
- A pretrained transformer (BERT or RoBERTa) for the heavy-lifting baseline.
- A tiny Streamlit app so anyone can paste a headline and get a verdict.
- Tracking how accuracy holds up over time or across topics.

Teamwork makes the model work.